# NB03 · Entrenamiento base YOLO11n para clasificación de madurez

**Entorno obligatorio:** GPU NVIDIA (T4 o L4)  
**No ejecutar con CPU:** el notebook se detendrá si CUDA no está disponible.  
**Tiempo estimado:** 20–40 min con T4; 15–25 min con L4.  
**Semilla:** 2026

Este notebook entrena un primer modelo reproducible para clasificar el racimo completo en cinco categorías:

`damaged`, `empty`, `overripe`, `ripe` y `unripe`.

Al finalizar guardará en Drive:

- pesos `best.pt` y `last.pt`;
- curvas de entrenamiento;
- métricas en el conjunto de prueba;
- predicciones por imagen;
- matrices de confusión absoluta y normalizada;
- una lámina con errores representativos.

> Antes de ejecutar: en Colab seleccione **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU** (o L4).


In [ ]:
# Instalación. La versión queda registrada más adelante para reproducibilidad.
%pip install -q -U ultralytics scikit-learn seaborn


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json
import platform
import random
import shutil
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import torch
import ultralytics
from PIL import Image, ImageOps
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from ultralytics import YOLO

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        'No se detectó GPU. En Colab seleccione Entorno de ejecución > '
        'Cambiar tipo de entorno de ejecución > T4 GPU y reinicie.'
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {gpu_name} ({gpu_memory_gb:.1f} GB)')
print('PyTorch:', torch.__version__)
print('Ultralytics:', ultralytics.__version__)


In [ ]:
PROJECT_ROOT = Path('/content/drive/MyDrive/PALMA_AFRICANA_YOLO')
DRIVE_DATA = PROJECT_ROOT / '02_DATOS_PROCESADOS' / 'classification_5class_v1'
MODELS_DIR = PROJECT_ROOT / '05_MODELOS'
RESULTS_ROOT = PROJECT_ROOT / '06_RESULTADOS'

RUN_NAME = 'YOLO11n_cls_seed2026'
DRIVE_MODEL_DIR = MODELS_DIR / RUN_NAME
DRIVE_RESULTS_DIR = RESULTS_ROOT / f'NB03_{RUN_NAME}'

LOCAL_DATA = Path('/content/PALMA_CLS')
LOCAL_RUNS = Path('/content/palma_runs')
LOCAL_RUN_DIR = LOCAL_RUNS / RUN_NAME

CLASSES = ['damaged', 'empty', 'overripe', 'ripe', 'unripe']
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

if not DRIVE_DATA.exists():
    raise FileNotFoundError(f'No se encontró {DRIVE_DATA}. Ejecute NB02 primero.')

for split in ['train', 'val', 'test']:
    split_dir = DRIVE_DATA / split
    if not split_dir.exists():
        raise FileNotFoundError(f'Falta el subconjunto: {split_dir}')
    missing = [label for label in CLASSES if not (split_dir / label).exists()]
    if missing:
        raise FileNotFoundError(f'Faltan clases en {split}: {missing}')

MODELS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_RUNS.mkdir(parents=True, exist_ok=True)

print('Datos de entrada:', DRIVE_DATA)
print('Modelo en Drive:', DRIVE_MODEL_DIR)
print('Resultados en Drive:', DRIVE_RESULTS_DIR)


In [ ]:
# Copiamos el conjunto al disco local de Colab para acelerar la lectura durante el entrenamiento.
if LOCAL_DATA.exists():
    shutil.rmtree(LOCAL_DATA)
shutil.copytree(DRIVE_DATA, LOCAL_DATA)

rows = []
for split in ['train', 'val', 'test']:
    for label in CLASSES:
        n = sum(
            1 for p in (LOCAL_DATA / split / label).iterdir()
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS
        )
        rows.append({'split': split, 'class': label, 'n': n})

distribution = pd.DataFrame(rows)
pivot = distribution.pivot(index='class', columns='split', values='n').fillna(0).astype(int)
display(pivot)

assert pivot['train'].sum() == 312, 'El total de entrenamiento no coincide con NB02.'
assert pivot['val'].sum() == 70, 'El total de validación no coincide con NB02.'
assert pivot['test'].sum() == 70, 'El total de prueba no coincide con NB02.'
assert (pivot > 0).all().all(), 'Alguna clase quedó vacía.'

distribution.to_csv(DRIVE_RESULTS_DIR / 'distribucion_usada.csv', index=False)
print('Copia local lista:', LOCAL_DATA)


## Configuración del experimento

Se usa transferencia de aprendizaje con `yolo11n-cls.pt`. Es un modelo pequeño apropiado como línea base. La selección del mejor punto se hace con el conjunto de validación; el conjunto de prueba permanece separado hasta la evaluación final.

La métrica central será **F1 macro**, porque el conjunto está desbalanceado y cada clase debe aportar el mismo peso a la evaluación.


In [ ]:
TRAIN_CONFIG = {
    'model': 'yolo11n-cls.pt',
    'epochs': 60,
    'imgsz': 384,
    'batch': 16,
    'patience': 15,
    'device': 0,
    'workers': 2,
    'seed': SEED,
    'deterministic': True,
    'pretrained': True,
    'optimizer': 'auto',
    'amp': True,
    'cache': True,
}

with open(DRIVE_RESULTS_DIR / 'configuracion_entrenamiento.json', 'w', encoding='utf-8') as f:
    json.dump(TRAIN_CONFIG, f, ensure_ascii=False, indent=2)

display(pd.Series(TRAIN_CONFIG, name='valor').to_frame())


In [ ]:
# Entrenamiento: esta es la celda que más tiempo demora.
start = time.time()
model = YOLO(TRAIN_CONFIG['model'])
train_results = model.train(
    data=str(LOCAL_DATA),
    epochs=TRAIN_CONFIG['epochs'],
    imgsz=TRAIN_CONFIG['imgsz'],
    batch=TRAIN_CONFIG['batch'],
    patience=TRAIN_CONFIG['patience'],
    device=TRAIN_CONFIG['device'],
    workers=TRAIN_CONFIG['workers'],
    seed=TRAIN_CONFIG['seed'],
    deterministic=TRAIN_CONFIG['deterministic'],
    pretrained=TRAIN_CONFIG['pretrained'],
    optimizer=TRAIN_CONFIG['optimizer'],
    amp=TRAIN_CONFIG['amp'],
    cache=TRAIN_CONFIG['cache'],
    project=str(LOCAL_RUNS),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    verbose=True,
)
elapsed_minutes = (time.time() - start) / 60

BEST_PT = LOCAL_RUN_DIR / 'weights' / 'best.pt'
LAST_PT = LOCAL_RUN_DIR / 'weights' / 'last.pt'
if not BEST_PT.exists():
    raise FileNotFoundError(f'No se generó el mejor peso esperado: {BEST_PT}')

print(f'Entrenamiento terminado en {elapsed_minutes:.1f} minutos.')
print('Mejor peso:', BEST_PT)


In [ ]:
# Evaluación independiente: una predicción por cada imagen de prueba.
best_model = YOLO(str(BEST_PT))
test_paths = sorted(
    p for p in (LOCAL_DATA / 'test').glob('*/*')
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
)
if len(test_paths) != 70:
    raise RuntimeError(f'Se esperaban 70 imágenes de prueba y se encontraron {len(test_paths)}.')

pred_results = best_model.predict(
    source=[str(p) for p in test_paths],
    imgsz=TRAIN_CONFIG['imgsz'],
    batch=TRAIN_CONFIG['batch'],
    device=0,
    verbose=False,
)
if len(pred_results) != len(test_paths):
    raise RuntimeError('El número de predicciones no coincide con el número de imágenes.')

prediction_rows = []
for path, result in zip(test_paths, pred_results):
    probs = result.probs.data.detach().cpu().numpy().astype(float)
    names = {int(k): v for k, v in result.names.items()}
    pred_id = int(np.argmax(probs))
    row = {
        'image': path.name,
        'relative_path': str(path.relative_to(LOCAL_DATA)),
        'true_class': path.parent.name,
        'pred_class': names[pred_id],
        'confidence': float(probs[pred_id]),
        'correct': path.parent.name == names[pred_id],
    }
    for class_id, probability in enumerate(probs):
        row[f'prob_{names[class_id]}'] = float(probability)
    prediction_rows.append(row)

predictions = pd.DataFrame(prediction_rows)
y_true = predictions['true_class']
y_pred = predictions['pred_class']

metrics = {
    'model': TRAIN_CONFIG['model'],
    'run_name': RUN_NAME,
    'seed': SEED,
    'gpu': gpu_name,
    'training_minutes': round(elapsed_minutes, 3),
    'n_test': int(len(predictions)),
    'accuracy': float(accuracy_score(y_true, y_pred)),
    'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
    'precision_macro': float(precision_score(y_true, y_pred, labels=CLASSES, average='macro', zero_division=0)),
    'recall_macro': float(recall_score(y_true, y_pred, labels=CLASSES, average='macro', zero_division=0)),
    'f1_macro': float(f1_score(y_true, y_pred, labels=CLASSES, average='macro', zero_division=0)),
    'f1_weighted': float(f1_score(y_true, y_pred, labels=CLASSES, average='weighted', zero_division=0)),
}

report = classification_report(
    y_true,
    y_pred,
    labels=CLASSES,
    target_names=CLASSES,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report).T

predictions.to_csv(DRIVE_RESULTS_DIR / 'predicciones_test.csv', index=False)
report_df.to_csv(DRIVE_RESULTS_DIR / 'reporte_clasificacion_test.csv')
with open(DRIVE_RESULTS_DIR / 'metricas_test.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

display(pd.Series(metrics, name='resultado').to_frame())
display(report_df.round(4))


In [ ]:
# Matrices de confusión para análisis científico y material del libro.
cm = confusion_matrix(y_true, y_pred, labels=CLASSES)
cm_norm = confusion_matrix(y_true, y_pred, labels=CLASSES, normalize='true')

pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(
    DRIVE_RESULTS_DIR / 'matriz_confusion_absoluta.csv'
)
pd.DataFrame(cm_norm, index=CLASSES, columns=CLASSES).to_csv(
    DRIVE_RESULTS_DIR / 'matriz_confusion_normalizada.csv'
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
axes[0].set_title('Matriz de confusión · frecuencias')
axes[0].set_xlabel('Clase predicha')
axes[0].set_ylabel('Clase real')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='YlGnBu', vmin=0, vmax=1,
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1])
axes[1].set_title('Matriz de confusión · normalizada por clase real')
axes[1].set_xlabel('Clase predicha')
axes[1].set_ylabel('Clase real')

for ax in axes:
    ax.tick_params(axis='x', rotation=35)
    ax.tick_params(axis='y', rotation=0)

cm_path = DRIVE_RESULTS_DIR / 'matrices_confusion_test.png'
fig.savefig(cm_path, dpi=220, bbox_inches='tight')
plt.show()
print('Guardado:', cm_path)


In [ ]:
# Lámina de errores con mayor confianza; si hay pocos, se completan con aciertos de menor confianza.
errors = predictions[~predictions['correct']].sort_values('confidence', ascending=False)
correct_low = predictions[predictions['correct']].sort_values('confidence')
examples = pd.concat([errors, correct_low], ignore_index=True).drop_duplicates('relative_path').head(12)

fig, axes = plt.subplots(3, 4, figsize=(14, 11))
for ax in axes.flat:
    ax.axis('off')

for ax, (_, row) in zip(axes.flat, examples.iterrows()):
    image_path = LOCAL_DATA / row['relative_path']
    with Image.open(image_path) as im:
        image = ImageOps.exif_transpose(im).convert('RGB')
        ax.imshow(image)
    color = 'darkgreen' if row['correct'] else 'crimson'
    ax.set_title(
        f"Real: {row['true_class']}\nPred.: {row['pred_class']} ({row['confidence']:.2f})",
        fontsize=9,
        color=color,
    )
    ax.axis('off')

fig.suptitle('Ejemplos de clasificación en el conjunto de prueba', fontsize=15)
fig.tight_layout()
examples_path = DRIVE_RESULTS_DIR / 'ejemplos_prediccion_test.png'
fig.savefig(examples_path, dpi=220, bbox_inches='tight')
plt.show()
print('Errores en prueba:', len(errors), 'de', len(predictions))


In [ ]:
# Copia final a Drive y registro completo del entorno.
shutil.copytree(LOCAL_RUN_DIR, DRIVE_MODEL_DIR, dirs_exist_ok=True)
shutil.copy2(BEST_PT, DRIVE_MODEL_DIR / 'best.pt')
if LAST_PT.exists():
    shutil.copy2(LAST_PT, DRIVE_MODEL_DIR / 'last.pt')

environment = {
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'ultralytics': ultralytics.__version__,
    'scikit_learn': sklearn.__version__,
    'gpu': gpu_name,
    'gpu_memory_gb': round(gpu_memory_gb, 2),
    'seed': SEED,
}
with open(DRIVE_RESULTS_DIR / 'entorno_ejecucion.json', 'w', encoding='utf-8') as f:
    json.dump(environment, f, ensure_ascii=False, indent=2)

summary = {
    **metrics,
    'best_model_drive': str(DRIVE_MODEL_DIR / 'best.pt'),
    'results_drive': str(DRIVE_RESULTS_DIR),
}
with open(DRIVE_RESULTS_DIR / 'resumen_nb03.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print('NB03 FINALIZADO CORRECTAMENTE')
print('Mejor modelo:', DRIVE_MODEL_DIR / 'best.pt')
print('Resultados:', DRIVE_RESULTS_DIR)
print(f"Accuracy={metrics['accuracy']:.4f} | F1 macro={metrics['f1_macro']:.4f}")


## Criterio para continuar

No se tomará una decisión solo con `accuracy`. Revise especialmente:

1. **F1 macro**, para dar igual importancia a las cinco clases;
2. **recall por clase**, en especial `damaged` y `empty`;
3. la matriz de confusión normalizada;
4. los errores de alta confianza.

Con esos resultados, NB04 comparará una mejora controlada: modelo de mayor capacidad, ajuste de aumentos o estrategia frente al desbalance. No se debe tocar el conjunto de prueba para escoger hiperparámetros.
